## 模型输入 Prompts

一个语言模型的提示是用户提供的一组指令或输入，用于引导模型的响应，帮助它理解上下文并生成相关和连贯的基于语言的输出，例如回答问题、完成句子或进行对话。


- Prompts
├─ PromptTemplate       #提示模板：参数化的模型输入， 普通文本提示模板，变量填充
├─ ChatPromptTemplate   # 聊天消息模板，组装 System/Human/AI 消息
└─ ExampleSelectors     #示例选择器：动态选择要包含在提示中的示例， 动态挑选少样本示例，嵌入提示词


## 提示模板 Prompt Templates

**Prompt Templates 提供了一种预定义、动态注入、模型无关和参数化的提示词生成方式，以便在不同的语言模型之间重用模板。**

一个模板可能包括指令、少量示例以及适用于特定任务的具体背景和问题。

通常，提示要么是一个字符串（LLMs），要么是一组聊天消息（Chat Model）。
拆成 4 个关键词：
    预定义
    提前写好固定提示文案骨架，不用每次业务代码里手写一大段 prompt。
    动态注入
    骨架留占位符 {变量}，运行时再填真实内容，不用手动拼字符串。
    参数化
    把可变内容抽象成输入变量，模板和业务数据彻底分开。
    模型无关
    模板写好后，不管接 OpenAI、智谱、本地开源模型，不用改模板，直接复用。

类继承关系:

```
BasePromptTemplate --> PipelinePromptTemplate

                       StringPromptTemplate --> PromptTemplate
                                                FewShotPromptTemplate
                                                FewShotPromptWithTemplates
                                                
                       BaseChatPromptTemplate --> AutoGPTPrompt
                                                  ChatPromptTemplate --> AgentScratchPadChatPromptTemplate



BaseMessagePromptTemplate --> MessagesPlaceholder

                              BaseStringMessagePromptTemplate --> ChatMessagePromptTemplate
                                                                  HumanMessagePromptTemplate
                                                                  AIMessagePromptTemplate
                                                                  SystemMessagePromptTemplate

PromptValue --> StringPromptValue
                ChatPromptValue
```

BasePromptTemplate：所有提示模板的顶层抽象父类
    所有整体提示模板的祖宗，统一规范：
        都有格式化渲染方法
        统一输出 PromptValue
        给上层 Chain/Agent 统一类型依赖
# 从它分出两大主线 + 一个流水线模板：
PipelinePromptTemplate → 流水线提示模板：把多个小模板按顺序拼接成一个完整 prompt，适合复杂任务拆多段模板组合。
StringPromptTemplate → 纯字符串模板分支
    专门生成普通字符串 prompt，给老式 BaseLLM 用
        PromptTemplate：最基础普通字符串模板，日常用得最多
        FewShotPromptTemplate：少样本提示模板，内置示例配置，做示例植入
        FewShotPromptWithTemplates：进阶少样本，支持嵌套其他模板
BaseChatPromptTemplate → 聊天消息模板分支
    专门生成消息列表 List [BaseMessage]，给 BaseChatModel 用
        ChatPromptTemplate：最常用聊天模板，组装系统 / 用户 / AI 消息
        AgentScratchPadChatPromptTemplate：给 Agent 专用，带中间思考过程的聊天模板
        AutoGPTPrompt：适配 AutoGPT 风格的智能代理提示模板


BaseMessagePromptTemplate：单条消息的模板（系统 / 用户 / AI 消息模板）
    作用：聊天里每一条角色消息，都由这些消息模板单独生成，再拼进 ChatPromptTemplate 里。


MessagesPlaceholder：消息占位符，专门用来塞历史对话消息列表

BaseStringMessagePromptTemplate：字符串转消息模板的父类
    ChatMessagePromptTemplate：通用角色消息模板
    HumanMessagePromptTemplate：用户消息模板
    AIMessagePromptTemplate：AI 回复消息模板
    SystemMessagePromptTemplate：系统人设消息模板


PromptValue：模板渲染完的结果包装类
    模板渲染完，不会直接返回字符串 / 消息列表，而是包装成 PromptValue
    统一返回类型，上层不用区分是字符串还是消息，框架自动适配模型。

两个子类：
    StringPromptValue：包装普通字符串，适配文本 LLM
    ChatPromptValue：包装消息列表，适配 ChatModel


```
模板骨架：BasePromptTemplate
├─ 纯字符串路线 → PromptTemplate / 少样本模板 → 输出字符串
├─ 聊天消息路线 → ChatPromptTemplate → 组装多条消息模板
└─ 流水线组合 → 多模板拼接

单条消息模板：BaseMessagePromptTemplate
→ 分出系统/用户/AI消息模板 + 历史消息占位符

渲染结果：PromptValue
→ 包装字符串 / 包装消息列表，统一适配两类模型

```

### 使用 PromptTemplate 类生成提升词

**通常，`PromptTemplate` 类的实例，使用Python的`str.format`语法生成模板化提示；也可以使用其他模板语法（例如jinja2）。**

LangChain 支持两种模板语法：
    str.format：轻量无依赖，适合简单模板；
    jinja2：支持条件、循环、嵌套，适合复杂工程化提示词。

#### 使用构造函数（Initializer）实例化 PromptTemplate

使用构造函数实例化 `prompt_template` 时必须传入参数：`input_variables` 和 `template`。

在生成提示过程中，会检查输入变量与模板字符串中的变量是否匹配，如果不匹配，则会引发异常；

In [1]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    template="你是{role}，请把{content}翻译成英文",
    input_variables=["role", "content"]
)

# 填充变量 使用 format 生成提示
prompt = template.format(role="翻译助手", content="我爱吃苹果")
print(prompt)


你是翻译助手，请把我爱吃苹果翻译成英文
